# R07 — Parser evaluation with the 7B model that actually ran in the user study

**CPU only. Needs `OPENROUTER_API_KEY` (about 40 short requests) for the LLM variants; everything else runs offline.**
Addresses reviewer comments on the conversational interface (**R2.7**: the study exercised the repair/fallback
workflow, not the parser; **R1**: parser accuracy is reported for a 3B model but the study used another one).

Round 1 evaluated `Qwen2.5-3B-Instruct` (local). The user-study logs show that the interface actually ran
`qwen/qwen-2.5-7b-instruct` through OpenRouter (see `R06`). This notebook closes that gap:

1. **Re-uses the committed benchmark** — the 40 labelled utterances, the canonical comparator and the semantic-
   equivalence rules (O1, O2, N1, N2) are *extracted from* `shared/09_conversational_interface_and_parser_eval.ipynb`
   rather than retyped. The deployed parser pipeline (`LLM_CONTRACT`, repair, regex overlay, schema validation,
   fallback) is *extracted from* `demo_app/app.py` via `ast`, so the code under test is the repository's code.
2. **Scores three variants** on the same 40 utterances: the deployed parser with the 7B model, the same parser plus a
   deterministic *relative-constraint* overlay (a proposed fix, post-processing only, so it re-uses the same LLM
   responses), and a regex-only baseline with the LLM disabled (offline).
3. **Reports the metric the `used_fallback` flag cannot see**: semantically wrong parses that were *accepted*.
4. **Checks the proposed fix against the 20 relative-phrased queries that participants typed** in the study (offline).

Caveats stated up front: the study-time prompt text is not in the repository, so the deployed prompt in `app.py` is
used (system message "Output ONLY valid JSON. No extra text.", `temperature = 0`). The relative overlay is designed
after looking at this benchmark, so its score on the 40 utterances is **not** held-out evidence; the participant-typed
queries in block R7.6 are the less biased check, and a fresh held-out set is needed before claiming it in the paper.

Set `R07_MOCK=1` to exercise the plumbing with a deterministic mock model (outputs go to `outputs/smoke/`; the numbers
are meaningless).

In [ ]:
# ============================================================
# C0) Common setup (identical in every review notebook)
# ============================================================
from pathlib import Path
import json, re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
try:
    display
except NameError:
    display = print

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "font.family": "serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "figure.dpi": 150, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
})
SINGLE_COL = (3.35, 2.35)
WIDE_COL = (6.85, 2.65)

# ------------------------------------------------------------
# Robust repo-root detection (CWD, its parents, then rglob)
# ------------------------------------------------------------
CWD = Path.cwd().resolve()


def _is_repo(p: Path) -> bool:
    return (p / "mvtec" / "outputs_patchcore").is_dir() and (p / "proditec" / "outputs_patchcore").is_dir()


REPO = next((p for p in [CWD, *CWD.parents] if _is_repo(p)), None)
if REPO is None:
    for base in [CWD, CWD.parent]:
        for hit in base.rglob("outputs_patchcore"):
            if _is_repo(hit.parent.parent):
                REPO = hit.parent.parent
                break
        if REPO:
            break
if REPO is None:
    raise FileNotFoundError("Could not locate the repository root (needs mvtec/ and proditec/). "
                            "Launch Jupyter from the repo or from review/.")
REVIEW = REPO / "review"
OUT = REVIEW / "outputs"
OUT_TABLES, OUT_FIGS = OUT / "tables", OUT / "figures"
for d in (OUT_TABLES, OUT_FIGS):
    d.mkdir(parents=True, exist_ok=True)

DS = {
    "mvtec": dict(root=REPO / "mvtec", pre="mvtec",
                  data="surrogate_ready_dataset/mvtec_4class_patchcore_surrogate_dataset.csv"),
    "proditec": dict(root=REPO / "proditec", pre="proditec",
                     data="surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv"),
}
for k, v in DS.items():
    r = v["root"]
    v["art"] = r / "surrogate_pkl_cfs_metadata"
    v["cfs"] = v["art"] / f"{k}_counterfactuals" / f"{k}_all_methods_cfs.csv"
    v["eval"] = v["art"] / f"{k}_counterfactuals_evaluation" / f"{k}_all_methods_cfs_patchcore_evaluation.csv"
    v["rev"] = v["art"] / f"{k}_revision_analysis"
    v["decomp"] = r / "revision_outputs" / "tables" / f"decomposition_per_candidate_{k}.csv"
    v["model"] = v["art"] / f"{k}_surrogate_catboost.cbm"
    v["meta"] = v["art"] / f"{k}_surrogate_metadata.json"
    v["dataset"] = r / v["data"]
    v["confirm"] = r / "revision_outputs" / "tables" / f"baseline_confirmation_{k}.csv"
    v["confirm_cands"] = r / "revision_outputs" / "tables" / f"baseline_confirm_candidates_{k}.csv"
print("REPO  :", REPO)
print("OUT   :", OUT)


def save_table(table: pd.DataFrame, stem: str, ff="%.4f"):
    p = OUT_TABLES / f"{stem}.csv"
    table.to_csv(p, index=False)
    try:
        table.to_latex(OUT_TABLES / f"{stem}.tex", index=False, escape=True, float_format=ff)
    except Exception as e:
        print("  (tex export skipped:", e, ")")
    print("Saved table:", p.relative_to(REPO))
    return p


def save_fig(fig, stem: str):
    for ext in ("pdf", "png"):
        fig.savefig(OUT_FIGS / f"{stem}.{ext}", bbox_inches="tight")
    plt.close(fig)
    print("Saved figure:", (OUT_FIGS / f"{stem}.pdf").relative_to(REPO))


def merge_numbers(section: str, values: dict):
    """Append to outputs/frozen_numbers.json (one section per notebook)."""
    p = OUT / "frozen_numbers.json"
    cur = json.loads(p.read_text()) if p.exists() else {}
    cur[section] = json.loads(json.dumps(values, default=float))
    p.write_text(json.dumps(cur, indent=2, sort_keys=True))
    print("Updated", p.relative_to(REPO), "->", section)

In [ ]:
# ============================================================
# R7.1) Configuration, extraction of the benchmark and of the deployed parser
# ============================================================
import ast, copy, hashlib, os, time
from scipy import stats

MOCK = os.environ.get("R07_MOCK", "0") == "1"
MODEL = os.environ.get("R07_MODEL", "qwen/qwen-2.5-7b-instruct")
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")     # never hard-code keys
if MOCK:                                                # plumbing runs never touch the real outputs
    OUT = OUT / "smoke"
    OUT_TABLES, OUT_FIGS = OUT / "tables", OUT / "figures"
    for _d in (OUT_TABLES, OUT_FIGS):
        _d.mkdir(parents=True, exist_ok=True)
CACHE_FILE = OUT / "cache" / ("R07_llm_cache_mock.json" if MOCK else "R07_llm_cache.json")
CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
LLM_CACHE = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}
print(f"model={MODEL} | mock={MOCK} | api key present={bool(API_KEY)} | cached responses={len(LLM_CACHE)}")

# ---- (a) benchmark + comparator from the committed parser-evaluation notebook ----
NB09 = REPO / "shared" / "09_conversational_interface_and_parser_eval.ipynb"
nb09 = json.loads(NB09.read_text())
src09 = ["".join(c["source"]) for c in nb09["cells"] if c["cell_type"] == "code"]
find = lambda needle: next(s for s in src09 if needle in s)

B = {}                                                  # namespace of the extracted benchmark code
exec(find("SOFT_REVIEW_BUDGETS = [i / 1000"), B)        # domain constants + q01 / snap_budget
B["TF_LO"], B["TF_HI"] = B["SOFT_TRAIN_FRACTION_RANGE"]
B["RB_LO"], B["RB_HI"] = min(B["SOFT_REVIEW_BUDGETS"]), max(B["SOFT_REVIEW_BUDGETS"])
exec("\n".join(l for l in find("LABELLED = [").splitlines() if "REV_OUT" not in l), B)
cmp_src = find("def canon_from_core")
cut_a, cut_b = cmp_src.index("# calibration"), cmp_src.index("# ---------------- semantic-equivalence")
exec(cmp_src[:cut_a] + cmp_src[cut_b:], B)              # comparator without the call into the HF parser
LABELLED = B["LABELLED"]
canon_from_core, canon_gold, field_match, semantic_row = (B[k] for k in
                                                          ("canon_from_core", "canon_gold", "field_match", "semantic_row"))
X_FACTUAL, Y_FACTUAL = B["X_FACTUAL_EVAL"], B["Y_FACTUAL_SUR_EVAL"]
print("labelled utterances:", len(LABELLED), "| x_factual:", X_FACTUAL, "| y_factual:", Y_FACTUAL)

# ---- (b) deployed parser from demo_app/app.py (pure functions only; app.py imports streamlit/torch) ----
APP = REPO / "demo_app" / "app.py"
tree = ast.parse(APP.read_text())
WANT = {"SOFT_TRAIN_FRACTION_RANGE", "SOFT_CORRUPTION_LEVELS", "SOFT_REVIEW_BUDGETS", "SOFT_REVIEW_BUDGET_BOUNDS",
        "DEFAULT_EPS", "DEFAULT_K", "MAX_LLM_RETRIES", "QUERY_SCHEMA", "LLM_CONTRACT", "snap_budget", "clamp", "snap01",
        "_extract_first_json", "_repair_objective", "_repair_soft_constraints", "_regex_overlay", "_fallback_query",
        "build_query"}
picked = []
for node in tree.body:
    names = ([t.id for t in node.targets if isinstance(t, ast.Name)] if isinstance(node, ast.Assign)
             else [node.target.id] if isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name)
             else [node.name] if isinstance(node, ast.FunctionDef) else [])
    if WANT & set(names):
        picked.append(node)
APPNS = {}
exec("import copy, json, re\nfrom typing import Any\nfrom jsonschema import validate\n", APPNS)
exec(compile(ast.Module(body=picked, type_ignores=[]), str(APP), "exec"), APPNS)
missing = WANT - set(APPNS)
assert not missing, f"not extracted from app.py: {missing}"
print("extracted from demo_app/app.py:", len(WANT), "objects | study-time app version was "
      "'2026-05-06-parser-valid-only-v2' (its source is not in the repo)")

In [ ]:
# ============================================================
# R7.2) LLM client (OpenRouter, temperature 0, cached) and mock
# ============================================================
import requests

SYSTEM_MSG = "Output ONLY valid JSON. No extra text."      # as in app.llm_generate_json
URL = "https://openrouter.ai/api/v1/chat/completions"


def _ckey(prompt):
    return hashlib.sha1((MODEL + "\n" + SYSTEM_MSG + "\n" + prompt).encode()).hexdigest()


def prompt_for(text):                                       # identical to app.build_query
    return APPNS["LLM_CONTRACT"] + "\n\nUSER REQUEST:\n" + text.strip()


def _save_cache():
    CACHE_FILE.write_text(json.dumps(LLM_CACHE, indent=1))


def openrouter_raw(prompt, tries=4):
    key = _ckey(prompt)
    if key in LLM_CACHE:
        return LLM_CACHE[key]
    if not API_KEY:
        raise RuntimeError("OPENROUTER_API_KEY not set and response not cached")
    last = None
    for i in range(tries):
        try:
            r = requests.post(URL, timeout=90, headers={"Authorization": f"Bearer {API_KEY}"},
                              json=dict(model=MODEL, temperature=0, max_tokens=400,
                                        messages=[{"role": "system", "content": SYSTEM_MSG},
                                                  {"role": "user", "content": prompt}]))
            r.raise_for_status()
            txt = r.json()["choices"][0]["message"]["content"]
            LLM_CACHE[key] = txt
            _save_cache()
            return txt
        except Exception as e:                              # transport errors are retried, never cached
            last = e
            time.sleep(2 * (i + 1))
    raise RuntimeError(f"OpenRouter request failed: {last}")


def llm_fn_real(prompt):                                    # same contract as app.llm_generate_json: JSON string out
    return APPNS["_extract_first_json"](openrouter_raw(prompt))


GOLD_BY_TEXT = {u: g for u, g in LABELLED}


def llm_fn_mock(prompt):
    """Deterministic stand-in: echoes the gold, fails on 'hard' items, and drops relative constraints like the real model."""
    text = prompt.split("USER REQUEST:\n", 1)[1]
    g = GOLD_BY_TEXT[text]
    if "hard" in g["tags"]:
        return "not json at all"
    o = g["objective"]
    obj = ({"type": "target_min", "value": o[1]} if o[0] == "target_min"
           else {"type": "delta_improve", "delta": o[1]} if o[0] == "delta_improve"
           else {"type": "target_band", "lower": o[1], "upper": o[2]})

    def num(s):
        if s[0] == "free": return {"mode": "free"}
        if s[0] == "fixed": return {"mode": "fixed", "value": s[1]}
        if s[0] == "range": return {"mode": "range", "lower": s[1], "upper": s[2]}
        return {"mode": "range", "lower": 0.5, "upper": 0.5}      # rel_max: the schema cannot express it
    c = g["corr"]
    corr = ({"mode": "free"} if c[0] == "free" else {"mode": "fixed", "value": c[1]} if c[0] == "fixed"
            else {"mode": "allowed", "allowed": c[1]})
    return json.dumps({"objective": obj, "soft_constraints": {
        "params_soft_train_fraction": num(g["tf"]), "params_soft_review_budget": num(g["rb"]),
        "params_soft_corruption_level": corr}, "selection": {"k": g["k"]}})


have_all = all(_ckey(prompt_for(u)) in LLM_CACHE for u, _ in LABELLED)
LLM_AVAILABLE = MOCK or bool(API_KEY) or have_all
LLM_FN = llm_fn_mock if MOCK else llm_fn_real
if not LLM_AVAILABLE:
    print("\n*** OPENROUTER_API_KEY is not set and the response cache is incomplete: the 7B variants are SKIPPED. ***\n"
          "    Export the key (export OPENROUTER_API_KEY=...) and re-run; only the offline blocks will be produced now.")
else:
    print("LLM variants will run:", "mock" if MOCK else ("cache only" if not API_KEY else "live API (+cache)"))

In [ ]:
# ============================================================
# R7.3) Relative-constraint overlay (proposed fix) and the three parser variants
# ============================================================
# The deployed schema has no way to express "10 % less review budget"; the model has to invent an absolute number and
# round-1 results show it does not. The overlay is deterministic post-processing of the parsed query: it only fires when
# the text contains a *reduction* word next to a percentage (or "a tenth") and mentions the review budget/effort.
REL_PCT = re.compile(r"(\d+(?:[.,]\d+)?)\s*(?:%|percent)\s*(?:less|lower|fewer|off|reduction)"
                     r"|(?:by|cut|cuts|reduce|reduced|reduces|reducing|shave|shaves|off)\b[^0-9%]{0,30}?(\d+(?:[.,]\d+)?)\s*(?:%|percent)"
                     r"|\bby a tenth\b|\ba tenth\b", re.I)
BUDGET_WORDS = re.compile(r"review|budget|effort|inspect", re.I)


def relative_reduction(text):
    """Fraction (0-1) by which the user asks the review budget to shrink, or None."""
    if not BUDGET_WORDS.search(text):
        return None
    m = REL_PCT.search(text)
    if not m:
        return None
    if m.group(0).lower().endswith("tenth"):
        return 0.10
    g = next(x for x in m.groups() if x)
    return float(g.replace(",", ".")) / 100


def rel_overlay(q, text, budget_anchor):
    p = relative_reduction(text)
    if p is None or not (0 < p < 1):
        return q
    q = copy.deepcopy(q)
    q["soft_constraints"]["params_soft_review_budget"] = {
        "mode": "range", "lower": APPNS["SOFT_REVIEW_BUDGETS"][0],
        "upper": APPNS["snap_budget"](budget_anchor * (1 - p))}
    q["_meta"]["relative_overlay"] = True
    return q


def parse_deployed(text):
    return APPNS["build_query"](text, X_FACTUAL, Y_FACTUAL, LLM_FN)


def parse_with_fix(text):
    return rel_overlay(parse_deployed(text), text, X_FACTUAL["params_soft_review_budget"])


def _no_llm(prompt):
    raise RuntimeError("LLM disabled")


def parse_regex_only(text):                                 # build_query with the LLM switched off = fallback + regex overlay
    return APPNS["build_query"](text, X_FACTUAL, Y_FACTUAL, _no_llm, max_retries=0)


def to_core(q):
    """demo_app query schema (value/lower/upper) -> the canonical schema the round-1 comparator reads (bounds)."""
    o = q["objective"]
    ob = ({"type": "target_min", "target_min": o["value"]} if o["type"] == "target_min"
          else {"type": "delta_improve", "delta": o["delta"]} if o["type"] == "delta_improve"
          else {"type": "target_band", "target_band": [o["lower"], o["upper"]]})
    sc = {}
    for k, c in q["soft_constraints"].items():
        c = dict(c)
        if c.get("mode") == "range":
            c["bounds"] = [c["lower"], c["upper"]]
        sc[k] = c
    return {"objective": ob, "soft_constraints": sc, "selection": q["selection"]}


FIELDS = ["obj", "tf", "rb", "corr", "k"]
VARIANTS = {
    "regex-only (LLM off)": (parse_regex_only, True),
    f"deployed parser, {MODEL.split('/')[-1]}": (parse_deployed, LLM_AVAILABLE),
    f"deployed + relative overlay, {MODEL.split('/')[-1]}": (parse_with_fix, LLM_AVAILABLE),
}


def evaluate(name, fn):
    rows = []
    for i, (utt, g) in enumerate(LABELLED):
        t0 = time.time()
        try:
            q = fn(utt)
            got, exp, fb = canon_from_core(to_core(q)), canon_gold(g), bool(q["_meta"]["used_fallback"])
            strict = {f: bool(field_match(got[f], exp[f])) for f in FIELDS}
            sem = semantic_row(got, exp, fb)
            err = ""
        except Exception as e:                              # an unparseable output must be counted, not dropped
            got = exp = None; fb = True; strict = {f: False for f in FIELDS}; sem = dict(strict); err = repr(e)[:120]
            exp = canon_gold(g)
        rows.append(dict(variant=name, idx=i, utterance=utt, tags=",".join(g["tags"]), used_fallback=fb,
                         overlay=bool(q["_meta"].get("relative_overlay")) if err == "" else False,
                         **{f"{f}_strict": strict[f] for f in FIELDS}, **{f"{f}_sem": bool(sem[f]) for f in FIELDS},
                         all_strict=all(strict.values()), all_sem=all(bool(v) for v in sem.values()),
                         got=str(got), expected=str(exp), error=err, seconds=round(time.time() - t0, 2)))
    return pd.DataFrame(rows)


PER = []
for name, (fn, ok) in VARIANTS.items():
    if not ok:
        print("skipped:", name)
        continue
    PER.append(evaluate(name, fn))
    print(f"{name:55s} strict={PER[-1].all_strict.mean():.3f}  semantic={PER[-1].all_sem.mean():.3f}  "
          f"fallback={PER[-1].used_fallback.mean():.3f}")
PER = pd.concat(PER, ignore_index=True)
save_table(PER.drop(columns=["seconds"]), "R07_parser_per_utterance")

In [ ]:
# ============================================================
# R7.4) Summary table (with Wilson intervals) and the "semantic error not flagged by used_fallback" metric
# ============================================================
def wilson(k, n):
    if n == 0:
        return (np.nan, np.nan)
    ci = stats.binomtest(int(k), int(n)).proportion_ci(method="wilson")
    return ci.low, ci.high


SUBSETS = {"all 40": lambda d: d.index == d.index,
           "relative-constraint": lambda d: d.tags.str.contains("relative"),
           "needs repair/out-of-domain": lambda d: d.tags.str.contains("repair_expected"),
           "decoy numbers": lambda d: d.tags.str.contains("decoy_numbers"),
           "paraphrases": lambda d: d.tags.str.contains("paraphrase"),
           "all others": lambda d: ~d.tags.str.contains("relative|repair_expected|decoy_numbers|paraphrase")}
rows = []
for v, d in PER.groupby("variant", sort=False):
    for sname, sel in SUBSETS.items():
        s = d[sel(d)]
        n = len(s)
        acc = s[~s.used_fallback]
        wrong_acc = int((~acc.all_sem).sum())
        lo, hi = wilson(s.all_sem.sum(), n)
        rows.append(dict(variant=v, subset=sname, n=n, fallback_rate=s.used_fallback.mean(),
                         strict_acc=s.all_strict.mean(), semantic_acc=s.all_sem.mean(), semantic_ci_lo=lo, semantic_ci_hi=hi,
                         accepted_n=len(acc), semantic_acc_among_accepted=acc.all_sem.mean() if len(acc) else np.nan,
                         accepted_but_wrong=wrong_acc,
                         accepted_but_wrong_share_of_all_errors=(wrong_acc / max(int((~s.all_sem).sum()), 1))))
SUMMARY = pd.DataFrame(rows)
display(SUMMARY[SUMMARY.subset.isin(["all 40", "relative-constraint"])].round(3))
save_table(SUMMARY, "R07_parser_summary")

# per-field semantic accuracy for the headline subset
FIELD_ACC = (PER.groupby("variant", sort=False)[[f"{f}_sem" for f in FIELDS]].mean()
             .rename(columns=lambda c: c.replace("_sem", "")).reset_index())
display(FIELD_ACC.round(3))
save_table(FIELD_ACC, "R07_parser_field_accuracy")

# committed round-1 numbers (Qwen2.5-3B, local) for direct comparison
OLD = shared_old = REPO / "shared" / "revision_outputs" / "parser_eval_v2_summary.csv"
if OLD.exists():
    old = pd.read_csv(OLD)
    print("\nCommitted round-1 parser_eval_v2_summary.csv (3B model):")
    display(old)

In [ ]:
# ============================================================
# R7.5) Errors that the fallback flag does not see: listing, and the relative-subset diagnostic
# ============================================================
if len(PER):
    ERR = PER[(~PER.used_fallback) & (~PER.all_sem)][["variant", "idx", "utterance", "tags", "got", "expected"]]
    print(f"accepted-but-wrong parses (not flagged by used_fallback): {len(ERR)}")
    display(ERR.head(25))
    save_table(ERR, "R07_accepted_but_wrong")

    REL = PER[PER.tags.str.contains("relative")][["variant", "utterance", "used_fallback", "rb_sem", "obj_sem", "all_sem", "got", "expected"]]
    save_table(REL, "R07_relative_subset")

    fig, ax = plt.subplots(figsize=WIDE_COL)
    sub = SUMMARY[SUMMARY.subset.isin(["all 40", "relative-constraint", "needs repair/out-of-domain", "paraphrases"])]
    vs = list(dict.fromkeys(sub.variant))
    subs = list(dict.fromkeys(sub.subset))
    w = 0.8 / len(vs)
    for j, v in enumerate(vs):
        y = [float(sub[(sub.variant == v) & (sub.subset == s)].semantic_acc.iloc[0]) for s in subs]
        ax.bar(np.arange(len(subs)) + j * w, y, w, label=v)
    ax.set_xticks(np.arange(len(subs)) + w * (len(vs) - 1) / 2)
    ax.set_xticklabels([f"{s}\n(n={int(sub[sub.subset == s].n.iloc[0])})" for s in subs])
    ax.set_ylabel("semantic accuracy (all fields)"); ax.set_ylim(0, 1.05)
    ax.legend(frameon=False, loc="upper right", ncol=1)
    save_fig(fig, "R07_parser_accuracy")

In [ ]:
# ============================================================
# R7.6) Offline check of the relative overlay on what participants really typed (study Task 3)
# ============================================================
# R06 holds the 72 typed queries. Task 3 asked for "F1 not lower than the base configuration and review budget reduced
# by at least 5 %" with a base budget of 0.375; the absolute equivalent is budget <= 0.35625. For each relative-phrased
# text we ask: does the overlay recognise a reduction and produce a bound that implies the gold (<= 0.35625)?
R06T = OUT_TABLES.parent / "tables" / "R06_parse_audit_per_attempt.csv"
if not R06T.exists():                                       # smoke/mock runs write elsewhere; fall back to the real R06 table
    R06T = REVIEW / "outputs" / "tables" / "R06_parse_audit_per_attempt.csv"
STUDY = {}
if R06T.exists():
    A6 = pd.read_csv(R06T)
    T3 = A6[(A6.task == "task_3") & (A6.relative_phrasing)].copy()
    T3["reduction_detected"] = T3.text.map(relative_reduction)
    T3["overlay_upper"] = T3.reduction_detected.map(lambda p: APPNS["snap_budget"](0.375 * (1 - p)) if pd.notna(p) else np.nan)
    T3["overlay_meets_gold"] = T3.overlay_upper <= 0.35625 + 1e-9
    T3["logged_zero_options"] = T3.n_options == 0
    print(f"relative-phrased Task-3 attempts: {len(T3)} | logged with 0 options: {int(T3.logged_zero_options.sum())}"
          f" | overlay recognises a reduction in {int(T3.reduction_detected.notna().sum())}"
          f" and yields a gold-compatible bound in {int(T3.overlay_meets_gold.sum())}")
    display(T3[["participant", "text", "n_options", "reduction_detected", "overlay_upper", "overlay_meets_gold"]].head(25))
    save_table(T3[["participant", "attempt", "text", "n_options", "reduction_detected", "overlay_upper", "overlay_meets_gold"]],
               "R07_overlay_on_study_task3")
    STUDY = dict(n_relative_phrased=len(T3), n_logged_zero_options=int(T3.logged_zero_options.sum()),
                 n_overlay_recognised=int(T3.reduction_detected.notna().sum()), n_overlay_meets_gold=int(T3.overlay_meets_gold.sum()))
    # precision guard: the overlay must not fire on non-relative study queries (tasks 1-2 and non-relative task 3)
    OTHER = A6[~((A6.task == "task_3") & (A6.relative_phrasing))]
    fp = OTHER[OTHER.text.map(relative_reduction).notna()]
    print(f"overlay false positives on the {len(OTHER)} non-relative study queries: {len(fp)}")
    STUDY["false_positives_on_nonrelative"] = len(fp)
else:
    print("R06 table not found - run R06 first; skipping.")

In [ ]:
# ============================================================
# R7.7) Freeze the numbers
# ============================================================
nums = dict(model=MODEL, mock=MOCK, llm_variants_run=bool(LLM_AVAILABLE), n_utterances=len(LABELLED), study_task3_overlay=STUDY)
if len(PER):
    for v, d in PER.groupby("variant", sort=False):
        nums[v] = dict(strict_acc=float(d.all_strict.mean()), semantic_acc=float(d.all_sem.mean()),
                       fallback_rate=float(d.used_fallback.mean()),
                       accepted_but_wrong=int(((~d.used_fallback) & (~d.all_sem)).sum()),
                       relative_semantic_acc=float(d[d.tags.str.contains("relative")].all_sem.mean()))
merge_numbers("R07_parser_eval" + ("_MOCK" if MOCK else ""), nums)
print(json.dumps(nums, indent=1, default=float))